# Threshold Lookup Evaluation

Purpose:
    This notebook inspects the deterministic threshold route in
    `llm_rag/iii_vector_db/threshold_lookup.py`.

Flow:
1. Define representative threshold-style and non-threshold prompts.
2. Check whether each prompt is detected as threshold-like.
3. Inspect the inferred criterion.
4. Display the deterministic answer returned by the lookup helper.

This notebook helps verify that stable numeric threshold facts are handled by
code rather than left entirely to retrieval and generation.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "llm_rag").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repo root containing 'llm_rag'.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
VECTOR_DB_DIR = REPO_ROOT / "llm_rag" / "iii_vector_db"
if str(VECTOR_DB_DIR) not in sys.path:
    sys.path.insert(0, str(VECTOR_DB_DIR))

from threshold_lookup import answer_threshold_query, infer_criterion, is_threshold_query

print(f"Repo root: {REPO_ROOT}")
print(f"Vector DB dir: {VECTOR_DB_DIR}")

Repo root: G:\My Drive\Documents\3. Education\7. Imperial\Modules\SWE Group Project\IUCN_Reviewer
Vector DB dir: G:\My Drive\Documents\3. Education\7. Imperial\Modules\SWE Group Project\IUCN_Reviewer\llm_rag\iii_vector_db


In [2]:
THRESHOLD_PROMPTS = [
    "What are the Criterion B thresholds for EOO and AOO?",
    "How many locations are allowed under Criterion B?",
    "What mature individual thresholds are used under Criterion D?",
    "What are the D2 thresholds?",
    "What extinction probability is used under Criterion E?",
    "What is the AOO threshold?",
    "What is the EOO threshold?",
    "Tell me about general species documentation.",
]

pd.DataFrame({"prompt": THRESHOLD_PROMPTS})

,prompt
0,What are the Criterion B thresholds for EOO an...
1,How many locations are allowed under Criterion B?
2,What mature individual thresholds are used und...
3,What are the D2 thresholds?
4,What extinction probability is used under Crit...
5,What is the AOO threshold?
6,What is the EOO threshold?
7,Tell me about general species documentation.


In [3]:
rows = []
for prompt in THRESHOLD_PROMPTS:
    is_match = is_threshold_query(prompt)
    criterion = infer_criterion(prompt)
    answer = answer_threshold_query(prompt)
    rows.append(
        {
            "prompt": prompt,
            "is_threshold_query": is_match,
            "inferred_criterion": criterion,
            "has_answer": answer is not None,
            "answer_preview": (answer or "")[:140].replace("\n", " "),
        }
    )

threshold_eval_df = pd.DataFrame(rows)
threshold_eval_df

,prompt,is_threshold_query,inferred_criterion,has_answer,answer_preview
0,What are the Criterion B thresholds for EOO an...,True,B,True,"Criterion B1 EOO thresholds: CR < 100 km², EN ..."
1,How many locations are allowed under Criterion B?,True,B,True,Typical number of locations thresholds under C...
2,What mature individual thresholds are used und...,True,D,True,Criterion D: Very small or restricted populati...
3,What are the D2 thresholds?,True,D,True,Criterion D2 thresholds: AOO < 20 km² or numbe...
4,What extinction probability is used under Crit...,True,E,True,Criterion E extinction probability thresholds:...
5,What is the AOO threshold?,True,B,True,"Criterion B2 AOO thresholds: CR < 10 km², EN <..."
6,What is the EOO threshold?,True,B,True,"Criterion B1 EOO thresholds: CR < 100 km², EN ..."
7,Tell me about general species documentation.,False,None,False,


In [4]:
for prompt in THRESHOLD_PROMPTS:
    answer = answer_threshold_query(prompt)
    display(Markdown(f"## Prompt\n\n`{prompt}`"))
    display(Markdown(f"- Threshold query: `{is_threshold_query(prompt)}`"))
    display(Markdown(f"- Inferred criterion: `{infer_criterion(prompt)}`"))
    if answer is None:
        display(Markdown("- Deterministic answer: `None`"))
    else:
        display(Markdown("### Deterministic Answer"))
        display(Markdown(answer.replace("\n", "  \n")))


## Prompt

`What are the Criterion B thresholds for EOO and AOO?`

- Threshold query: `True`

- Inferred criterion: `B`

### Deterministic Answer

Criterion B1 EOO thresholds: CR < 100 km², EN < 5,000 km², VU < 20,000 km².  
Criterion B2 AOO thresholds: CR < 10 km², EN < 500 km², VU < 2,000 km².

...

Criterion B1 EOO thresholds: CR < 100 km², EN < 5,000 km², VU < 20,000 km².

## Prompt

`Tell me about general species documentation.`

- Threshold query: `False`

- Inferred criterion: `None`

- Deterministic answer: `None`